<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/AdvancedAgent/ipynb/bonus_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-openai

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

os.makedirs(name="resources", exist_ok=True)
_ = load_dotenv(dotenv_path=".env", override=True)

In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [4]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:
    """
    Obtain information from the database using SQL queries
    """
    try:
        return db.run(command=query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke(input="SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [5]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gpt-oss:120b-cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[sql_query]
)

In [7]:
import time
from langchain.messages import HumanMessage

question = HumanMessage(content="""
    Who is the most popular artist beginning with 'S' in this database?
""")

start_time = time.time()
response = agent.invoke(
    {"messages": [question]}
)
end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

Time taken: 4.97s


In [8]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================


    Who is the most popular artist beginning with 'S' in this database?

================================== Ai Message ==================================
Tool Calls:
  sql_query (call_wp21yc05)
 Call ID: call_wp21yc05
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table';
================================= Tool Message =================================
Name: sql_query

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]
================================== Ai Message ==================================
Tool Calls:
  sql_query (call_4tw8ejql)
 Call ID: call_4tw8ejql
  Args:
    query: SELECT ar.Name, SUM(il.UnitPrice * il.Quantity) AS total_sales
FROM Artist ar
JOIN Album al ON ar.ArtistId = al.ArtistId
JOIN Track tr ON al.AlbumId = tr.AlbumId
JOIN InvoiceLine il ON tr.TrackId = il.Tra

In [9]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT ar.Name, SUM(il.UnitPrice * il.Quantity) AS total_sales
FROM Artist ar
JOIN Album al ON ar.ArtistId = al.ArtistId
JOIN Track tr ON al.AlbumId = tr.AlbumId
JOIN InvoiceLine il ON tr.TrackId = il.TrackId
WHERE ar.Name LIKE 'S%'
GROUP BY ar.ArtistId
ORDER BY total_sales DESC
LIMIT 1;
